# Project Setup

## Overview
This notebook is part of a reproducible GRPO-based fine-tuning pipeline for cybersecurity policy generation.

## Purpose of this setup cell
The first code cell in this notebook:
- loads environment variables from `.env`
- identifies the project root directory automatically
- defines standard folder paths used across the project
- creates required folders if they do not already exist

## Why this matters
This makes the notebook portable across macOS, Windows, and Linux without requiring users to manually edit file paths.

## Expected project folders
- `data/corpus/` → source PDF corpus
- `data/processed/` → intermediate processed data
- `data/sample/` → optional small example data
- `outputs/completions/` → generated completions
- `outputs/rankings/` → ranked outputs
- `outputs/models/` → trained model checkpoints
- `outputs/evaluations/` → evaluation results

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Define PROJECT_ROOT automatically
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

# Define folders
DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "corpus"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
COMPLETIONS_DIR = OUTPUT_DIR / "completions"
RANKINGS_DIR = OUTPUT_DIR / "rankings"
MODELS_DIR = OUTPUT_DIR / "models"
EVAL_DIR = OUTPUT_DIR / "evaluations"

# Create folders if needed
for folder in [
    DATA_DIR, CORPUS_DIR, PROCESSED_DIR,
    OUTPUT_DIR, COMPLETIONS_DIR, RANKINGS_DIR, MODELS_DIR, EVAL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

# Step 5 — Evaluate the Base Model

## Purpose
This notebook evaluates the model outputs using the project’s defined metrics.

## Why this step is important
Evaluation provides an objective view of model quality and establishes a baseline for comparison with the fine-tuned system.

## Inputs
- Ranked completion data from `outputs/rankings/`

## Outputs
- Evaluation metrics saved to `outputs/evaluations/`

## Main tasks
1. Load the ranked data
2. Compute rubric-based summary metrics
3. Calculate preference-related statistics
4. Save results for reporting and comparison

## Success criteria
This step is complete when the evaluation metrics are computed successfully and written to the evaluations output folder.

In [1]:
#  GRPO Base Model Metrics Calculator
# ==============================================

import json
import pandas as pd
from statistics import mean, stdev
from collections import Counter
from itertools import combinations
from typing import List

# ===  Step 1: Load JSONL file ===
jsonl_path = '/content/cyber_policies_4comps_grpo_ranked.jsonl'

def load_jsonl(filepath):
    with open(filepath, 'r') as f:
        return [json.loads(line) for line in f]

data = load_jsonl(jsonl_path)

# === Step 2: Initialize counters ===
rubric_scores = []
pairwise_preferences = []
margin_values = []

for item in data:
    scores = item.get("scores", [])
    if not scores or len(scores) < 2:
        continue

    # Extract rubric totals
    totals = [s["total"] for s in scores]
    rubric_scores.extend(totals)

    # Pairwise preferences (for preference accuracy)
    for pair in combinations(scores, 2):
        score_a = pair[0]["total"]
        score_b = pair[1]["total"]
        idx_a = pair[0]["index"]
        idx_b = pair[1]["index"]

        # Establish pairwise preference
        if score_a > score_b:
            pairwise_preferences.append((idx_a, idx_b))  # a > b
        elif score_b > score_a:
            pairwise_preferences.append((idx_b, idx_a))  # b > a

        margin = abs(score_a - score_b)
        margin_values.append(margin)

# === Step 3: Calculate Metrics ===
rubric_mean = round(mean(rubric_scores), 3)
rubric_std = round(stdev(rubric_scores), 3)

# Count how many pairwise preferences match ground truth preferences
gt_preferences = []
for item in data:
    if "preferences" in item:
        gt_preferences.extend([(p["winner"], p["loser"]) for p in item["preferences"]])

correct_pref = sum(1 for pref in pairwise_preferences if pref in gt_preferences)
preference_accuracy = round(correct_pref / len(pairwise_preferences), 3) if pairwise_preferences else 0

margin_mean = round(mean(margin_values), 3)
margin_std = round(stdev(margin_values), 3)

# === Step 4: Print Summary ===
print("GRPO Evaluation Metrics for Base Model")
print("========================================")
print(f"Rubric Score Mean: {rubric_mean}")
print(f"Rubric Score Std Dev: {rubric_std}")
print(f"Preference Accuracy: {preference_accuracy}")
print(f"Margin of Victory Mean: {margin_mean}")
print(f"Margin of Victory Std Dev: {margin_std}")
print(f"KL Divergence: N/A (requires base vs. fine-tuned)")
print(f"Win Rate vs Ref: N/A (requires direct model comparison)")


GRPO Evaluation Metrics for Base Model
Rubric Score Mean: 9.523
Rubric Score Std Dev: 2.0
Preference Accuracy: 1.0
Margin of Victory Mean: 2.576
Margin of Victory Std Dev: 1.745
KL Divergence: N/A (requires base vs. fine-tuned)
Win Rate vs Ref: N/A (requires direct model comparison)
